# 04 — Model Interpretation with SHAP Values

This notebook uses SHAP (SHapley Additive exPlanations) to interpret the best QSAR model.

**Key questions:**
- Which molecular features drive EGFR inhibitory activity?
- How does each feature contribute to individual predictions?
- Are the learned patterns chemically meaningful?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
RANDOM_STATE = 42

# Load Morgan FP data (best performing descriptor set)
loaded = np.load(DATA_DIR / "descriptors_morgan.npz", allow_pickle=True)
X = loaded["X"]; y_reg = loaded["y_reg"]; y_cls = loaded["y_cls"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y_reg, test_size=0.2,
                                            random_state=RANDOM_STATE,
                                            stratify=y_cls)
feature_names = [f"Bit_{i}" for i in range(X.shape[1])]
print(f"Train: {len(X_tr)} | Test: {len(X_te)}")

In [ ]:
# Train best model: XGBoost
model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=2, random_state=RANDOM_STATE, verbosity=0
)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)
from sklearn.metrics import r2_score
print(f"Test R² = {r2_score(y_te, y_pred):.3f}")

In [ ]:
# Compute SHAP values using TreeExplainer
explainer = shap.TreeExplainer(model)
X_shap = X_te[:500]  # subsample for speed
shap_values = explainer.shap_values(X_shap)
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# SHAP Summary Plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                  max_display=20, show=False)
plt.title("SHAP Summary Plot — XGBoost + Morgan FP\n(pIC50 prediction, EGFR)",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plots/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP Bar Plot (mean |SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[-20:][::-1]
top_names = [feature_names[i] for i in top_idx]
top_vals = mean_abs_shap[top_idx]

fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(top_idx)))
ax.barh(range(len(top_idx)), top_vals[::-1], color=colors[::-1])
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel("Mean |SHAP value|", fontsize=11)
ax.set_title("Top 20 Most Important Morgan FP Bits\n(XGBoost, pIC50 prediction)", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plots/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP Waterfall for a single high-activity compound
high_act_idx = np.argmax(y_te[:500])
shap_single = shap_values[high_act_idx]
sorted_idx = np.argsort(np.abs(shap_single))[-15:]
sorted_shap = shap_single[sorted_idx]
sorted_names = [feature_names[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(8, 6))
colors_wf = ["#E53935" if v > 0 else "#1E88E5" for v in sorted_shap]
ax.barh(range(len(sorted_idx)), sorted_shap, color=colors_wf, alpha=0.8)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(sorted_names, fontsize=9)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("SHAP value", fontsize=11)
ax.set_title(f"SHAP Waterfall — Single Compound\n(pIC50={y_te[high_act_idx]:.2f})", fontsize=11)
red_patch = mpatches.Patch(color="#E53935", label="Increases pIC50")
blue_patch = mpatches.Patch(color="#1E88E5", label="Decreases pIC50")
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plots/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Compound pIC50: {y_te[high_act_idx]:.2f}")
print(f"Predicted pIC50: {model.predict(X_te[high_act_idx:high_act_idx+1])[0]:.2f}")

In [ ]:
# Summary: key findings
print("=" * 50)
print("KEY FINDINGS FROM SHAP ANALYSIS")
print("=" * 50)
print(f"Total features: {X.shape[1]}")
print(f"Features with mean|SHAP| > 0.01: {(mean_abs_shap > 0.01).sum()}")
print(f"Top feature: {feature_names[top_idx[0]]} (mean|SHAP|={top_vals[0]:.4f})")
print(f"Top 10 features explain {100*top_vals[:10].sum()/mean_abs_shap.sum():.1f}% of total importance")